## Create state population reference table (managed table)

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "cfpb_dev")
catalog = dbutils.widgets.get("catalog")
raw_path = f"/Volumes/{catalog}/bronze/reference/acs_state_population.csv"
target = f"{catalog}.reference.state_population"

# Read everything as strings so codes like "010" and "01" keep their leading zeros
raw = spark.read.option("header", "true").csv(raw_path)

# FIPS -> USPS code (matches CFPB's 'state' field)
fips_to_usps = {
 "01":"AL","02":"AK","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT","10":"DE","11":"DC","12":"FL",
 "13":"GA","15":"HI","16":"ID","17":"IL","18":"IN","19":"IA","20":"KS","21":"KY","22":"LA","23":"ME",
 "24":"MD","25":"MA","26":"MI","27":"MN","28":"MS","29":"MO","30":"MT","31":"NE","32":"NV","33":"NH",
 "34":"NJ","35":"NM","36":"NY","37":"NC","38":"ND","39":"OH","40":"OK","41":"OR","42":"PA","44":"RI",
 "45":"SC","46":"SD","47":"TN","48":"TX","49":"UT","50":"VT","51":"VA","53":"WA","54":"WV","55":"WI",
 "56":"WY","72":"PR"
}
mapping = spark.createDataFrame(list(fips_to_usps.items()), ["state_fips", "state_code"])

states = (raw
    .filter(F.col("SUMLEV") == "040")                       # states (+ DC, PR if present)
    .select(F.col("STATE").alias("state_fips"),
            F.col("NAME").alias("state_name"),
            F.col("POPESTIMATE2025").cast("bigint").alias("population"))
    .withColumn("year", F.lit(2025))
    .withColumn("source", F.lit("Census PEP Vintage 2025"))
    .join(mapping, "state_fips", "left")
    .select("state_code", "state_fips", "state_name", "year", "population", "source"))

# --- Validation ---
us_total = int(raw.filter(F.col("SUMLEV") == "010").first()["POPESTIMATE2025"])
states_total = states.filter(F.col("state_code") != "PR").agg(F.sum("population")).first()[0]
unmapped = states.filter(F.col("state_code").isNull()).count()

print(f"State rows: {states.count()} (expect 51, or 52 with PR)")
print(f"Sum of states + DC: {states_total:,} | US total: {us_total:,} | match: {states_total == us_total}")
print(f"Unmapped FIPS codes: {unmapped}")
assert unmapped == 0 and states_total == us_total, "Validation failed. Check before writing."

# --- Write managed table (no LOCATION = managed) ---
states.write.mode("overwrite").saveAsTable(target)
spark.sql(f"COMMENT ON TABLE {target} IS 'State population, Census Population Estimates (Vintage 2025). Source file in bronze.reference volume.'")
display(spark.table(target).orderBy("state_code"))